In [ ]:
import os
import requests
import json
from pathlib import Path

ACCESS_KEY = ""
HEADERS = {"Authorization": f"Client-ID {ACCESS_KEY}"}

IMAGES_DIR = Path("images")
DATA_DIR = Path("data")
IMAGES_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

CATEGORIES = {
    "animaux": "animals",
    "batiments": "architecture",
    "paysage": "landscape",
}

metadata_list = []

for category_name, query in CATEGORIES.items():
    print(f"Chargement de la catégorie {category_name}...")

    url = "https://api.unsplash.com/search/photos"
    params = {
        "query": query,
        "per_page": 33,
        "page": 1,
        "orientation": "landscape",
    }

    response = requests.get(url, headers=HEADERS, params=params, timeout=30)
    response.raise_for_status()
    results = response.json()["results"]

    for i, photo in enumerate(results[:33], start=1):
        photo_id = photo["id"]

        url_photo = f"https://api.unsplash.com/photos/{photo_id}"
        response_photo = requests.get(url_photo, headers=HEADERS, timeout=30)
        response_photo.raise_for_status()
        photo_full = response_photo.json()

        # Nom du fichier
        filename = f"{category_name}_{photo_id}.jpg"
        filepath = IMAGES_DIR / filename

        # URL de l’image (regular ou full)
        image_url = photo_full["urls"]["regular"]

        # Téléchargement de l’image
        image_response = requests.get(image_url, stream=True, timeout=30)
        image_response.raise_for_status()

        with open(filepath, "wb") as f:
            for chunk in image_response.iter_content(chunk_size=8192):
                f.write(chunk)

        # Taille du fichier en Ko
        file_stat = filepath.stat()
        file_size_kb = round(file_stat.st_size / 1024, 2)

        # Metadonnées EXIF
        exif = photo_full.get("exif", {}) or {}
        make = exif.get("make") or ""
        model = exif.get("model") or ""
        camera = (make + " " + model).strip()
        aperture = exif.get("aperture")
        focal_length = exif.get("focal_length")
        iso = exif.get("iso")
        exposure = exif.get("exposure_time")

        taken_at = exif.get("created_at", "") or photo_full.get("created_at", "")

        licenses = photo_full.get("current_user_collections", [])
        license_str = "Unsplash license (check terms)"

        meta = {
            "nom_fichier": filename,
            "categorie": category_name,
            "largeur": photo_full["width"],
            "hauteur": photo_full["height"],
            "format_fichier": "jpg",
            "taille_fichier_kb": file_size_kb,
            "url_source": photo_full["urls"]["full"],
            "licence": license_str,
            "description": photo_full.get("description") or photo_full.get("alt_description"),
            "photographe": photo_full["user"]["name"],
            "page_unsplash": photo_full["links"]["html"],
            "date_creation_photo": photo_full["created_at"],
            "nombre_likes": photo_full.get("likes", 0),
            "exif": {
                "camera_model": camera if camera else None,
                "taken_at": taken_at if taken_at else None,
                "aperture": aperture,
                "focal_length": focal_length,
                "iso": iso,
                "exposure_time": exposure,
            },
        }

        metadata_list.append(meta)

# Sauvegarde des métadonnées
output_path = DATA_DIR / "images_metadata.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(metadata_list, f, ensure_ascii=False, indent=2)

print(f"✅ {len(metadata_list)} images et métadonnées sauvegardées.")

Chargement de la catégorie animaux...
Chargement de la catégorie batiments...
Chargement de la catégorie paysage...
✅ 9 images et métadonnées sauvegardées.
